# RAM Insight — เครื่องมือ Python ตรวจและเตรียมข้อมูล (Back-end)

เครื่องมือนี้เป็น "หลังบ้าน" ของเว็บ RAM Insight ทำ 3 อย่าง
1. **ตรวจคุณภาพข้อมูล (Data Quality Gate)** — คัดแถวที่เวลาขาดหาย / เวลาซ่อมเสร็จก่อนเริ่มเสีย / หยุดนานเกิน 1 ปี ออก
2. **คำนวณ MTBF, MTTR, Availability (และ R, M)** — ใช้สูตรและเกณฑ์เดียวกับเว็บทุกตัว
3. **ส่งออกไฟล์ Excel ที่สะอาดแล้ว** — เอาไปอัปโหลดเข้าเว็บ RAM Insight ได้ทันที

## วิธีใช้ (กดทีละเซลล์ จากบนลงล่าง)
1. กดปุ่ม ▶ ที่เซลล์ **"ขั้นที่ 1"** (รอให้ขึ้นเครื่องหมายถูกสีเขียว)
2. กดปุ่ม ▶ ที่เซลล์ **"ขั้นที่ 2"** จะมีปุ่ม *Choose Files* ให้เลือกไฟล์ Failure Log (.xlsx) จากเครื่อง
3. ดูผลที่ขึ้นด้านล่าง (ตรวจคุณภาพ + ตารางสรุป)
4. กดปุ่ม ▶ ที่เซลล์ **"ขั้นที่ 3"** เพื่อดาวน์โหลดไฟล์ที่สะอาดแล้ว

> **เรื่องข้อมูลจริง:** ถ้าไฟล์เป็นข้อมูลจริงของ กฟน. ที่เป็นความลับ การอัปโหลดขึ้น Colab คือการส่งไฟล์ไปที่เซิร์ฟเวอร์ Google
> ให้ขออนุญาตก่อน หรือใช้ข้อมูลจำลองแทน (โค้ดชุดเดียวกันรันบนเครื่องตัวเองได้ด้วย)


## ขั้นที่ 1 — โหลดกฎและสูตร (ไม่ต้องแก้อะไร)
โค้ดในเซลล์ถัดไปคือกฎชุดเดียวกับที่เขียนไว้ในหน้าเว็บ (ชื่อคอลัมน์ที่รองรับ, รูปแบบวันที่, เงื่อนไขตัดข้อมูล, สูตร R/M/A)

In [ ]:
import re, math, datetime as dt
import openpyxl

# ---- 1) ตารางชื่อคอลัมน์ที่ยอมรับ (คัดลอกจาก FIELD_ALIASES ในเว็บ ตรงตัว) ----
ALIASES = {
    'equipment_group': ['equipmentgroup','group','plant','area','site','ระบบ','กลุ่มอุปกรณ์','equipment'],
    'component': ['component','equipment','equipmentname','asset','item','part','subsystem','อุปกรณ์','ชื่ออุปกรณ์'],
    'code': ['code','id','tagno','tag','itemcode','recordid','no','failurecode','รหัส','รหัสอุปกรณ์'],
    'description': ['description','desc','detail','details','issue','problem','finding','symptom','failuremode','mode','รายละเอียด','อาการ','ลักษณะปัญหา'],
    'problem_category': ['problemcategory','category','issuetype','problemtype','type','ประเภทปัญหา','หมวดหมู่'],
    'cause': ['cause','rootcause','reason','failurecause','สาเหตุ'],
    'remedy': ['remedy','action','correctiveaction','solution','recommendation','การแก้ไข','วิธีแก้ไข'],
    'department': ['department','dept','responsible','owner','team','หน่วยงาน','แผนก'],
    'ram_classification': ['ramclassification','ramclass','ramcategory','ram'],
    'mtbf': ['mtbf','mtbfhrs','mtbfhours'],
    'mttr': ['mttr','mttrhrs','mttrhours'],
    'availability': ['availability','availabilitypct','avail','ความพร้อมใช้งาน'],
    'substation': ['substation','sub','station','สถานี','สถานีไฟฟ้า','สถานีย่อย'],
    'transformer_unit': ['transformerid','transformerunit','trno','หม้อแปลง','เลขหม้อแปลง','รหัสหม้อแปลง'],
    'mva': ['mva','ratingmva','ขนาดmva','ขนาดหม้อแปลง'],
    'failure_start': ['failurestart','startdatetime','starttime','failuretime','เวลาเริ่มขัดข้อง','วันเวลาเริ่มขัดข้อง','เวลาเริ่มต้นขัดข้อง','วันที่เริ่มขัดข้อง'],
    'repair_complete': ['repaircomplete','enddatetime','endtime','completetime','เวลาซ่อมเสร็จ','วันเวลาซ่อมเสร็จ','เวลาที่ซ่อมเสร็จ','วันที่ซ่อมเสร็จ','เวลาซ่อมแซมเสร็จ'],
}

# ---- 2) ค่ากำหนดและเกณฑ์ (เหมือนเว็บ) ----
PROFILES = {'monthly': (720, 12), 'yearly': (8760, 24)}   # (T_MISSION, T_REPAIR)
R_STD, M_STD, A_STD = 85, 50, 96                           # เกณฑ์ผ่าน (%)
MAX_DOWNTIME_HRS = 8760                                    # เกิน 1 ปี = ค่าผิดปกติ

def norm_key(k):
    return re.sub(r'[^a-z0-9\u0e01-\u0e59]', '', str(k).lower())

def js_str(v):
    if isinstance(v, float) and v.is_integer():
        v = int(v)
    return str(v).strip()

def js_parse_float(v):
    if isinstance(v, bool): return float('nan')
    if isinstance(v, (int, float)): return float(v)
    m = re.match(r'^\s*[-+]?(\d+\.?\d*|\.\d+)([eE][-+]?\d+)?', str(v))
    return float(m.group(0)) if m else float('nan')

# ---- 3) แปลงวันเวลา (เหมือน parseFlexibleDate) ----
_DMY = re.compile(r'^(\d{1,2})[/\-](\d{1,2})[/\-](\d{4})(?:[ T](\d{1,2}):(\d{2})(?::(\d{2}))?)?$')

def parse_flexible_date(val):
    if val is None or val == '':
        return None
    if isinstance(val, dt.datetime):
        return val
    if isinstance(val, dt.date):
        return dt.datetime(val.year, val.month, val.day)
    if isinstance(val, bool):
        return None
    if isinstance(val, (int, float)):            # เลขลำดับวันที่ของ Excel
        try:
            return dt.datetime(1970, 1, 1) + dt.timedelta(milliseconds=round((val - 25569) * 86400 * 1000))
        except (OverflowError, ValueError):
            return None
    s = str(val).strip()
    if not s:
        return None
    m = _DMY.match(s)
    if m:
        d, mo, y, h, mi, sec = m.groups()
        d, mo, y = int(d), int(mo), int(y)
        y += (mo - 1) // 12; mo = (mo - 1) % 12 + 1      # JS ยอมให้เดือน/วันเกินแล้วเลื่อนต่อ
        try:
            return dt.datetime(y, mo, 1) + dt.timedelta(days=d - 1, hours=int(h or 0), minutes=int(mi or 0), seconds=int(sec or 0))
        except (OverflowError, ValueError):
            return None
    try:
        return dt.datetime.fromisoformat(s.replace('Z', ''))
    except ValueError:
        return None

# ---- 4) อ่านแถว → รายการ (เหมือน mapRow / classifyEquipment / จำแนก R-A-M) ----
def normalize_sub(v):                          # เหมือน normalizeSub ในเว็บ
    t = '' if v is None else js_str(v).strip()
    if not t: return ''
    l = t.lower()
    if 'ฉลอง' in l or 'chalong' in l: return 'ฉลองกรุง'
    if 'ลาดกระบัง' in l or 'lat krabang' in l or 'ladkrabang' in l: return 'ลาดกระบัง'
    return t

def map_row(raw, sheet):
    norm = {}
    for k, v in raw.items():
        norm[norm_key(k)] = v
    out = {}
    for field, names in ALIASES.items():
        val = None
        for a in names:
            if a in norm and norm[a] is not None and norm[a] != '':
                val = norm[a]; break
        out[field] = val
    g = lambda f: js_str(out[f]) if out[f] not in (None, '', 0) else ''
    return {
        'sheet': sheet,
        'equipment_group': g('equipment_group'), 'component': g('component'),
        'substation': normalize_sub(out['substation']), 'unit': g('transformer_unit'),
        'code': g('code'), 'description': g('description'),
        'problem_category': g('problem_category') or (sheet or ''),
        'cause': g('cause'), 'remedy': g('remedy'), 'department': g('department'),
        'ram_raw': g('ram_classification'),
        'mtbf': js_parse_float(out['mtbf']) if out['mtbf'] is not None else float('nan'),
        'mttr': js_parse_float(out['mttr']) if out['mttr'] is not None else float('nan'),
        'availability_file': js_parse_float(out['availability']) if out['availability'] is not None else float('nan'),
        'failure_start': parse_flexible_date(out['failure_start']),
        'repair_complete': parse_flexible_date(out['repair_complete']),
    }

def equip_category(r):
    if r['component']:
        return r['component'].replace('_', ' ').replace('/', ' / ').strip()
    return 'ไม่ระบุ'

def parse_ram(raw):
    if not raw: return None
    low = raw.lower(); primary = None
    if 'reliability' in low: primary = 'Reliability'
    elif 'availability' in low: primary = 'Availability'
    elif 'maintainability' in low: primary = 'Maintainability'
    if not primary:
        t = raw.strip().upper()
        full = {'R': 'Reliability', 'A': 'Availability', 'M': 'Maintainability'}.get(t)
        return (full, '-', 'จากไฟล์') if full else None
    sub = re.sub(primary, '', raw, count=1, flags=re.I)
    sub = re.sub(r'^[\s/:\-()]+', '', sub); sub = re.sub(r'[)\s]+$', '', sub).strip() or '-'
    return (primary, sub, 'จากไฟล์')

def auto_ram(r):
    text = ((r['description'] or '') + ' ' + (r['cause'] or '')).lower()
    has = lambda *w: any(x in text for x in w)
    if has('รั่ว', 'leak') or (has('น้ำมัน') and has('ต่ำ', 'ระดับ')) or has('low level'):
        return ('Availability', 'รั่วซึม/ระดับต่ำ', 'อัตโนมัติ')
    if has('สนิม', 'เสื่อมสภาพ', 'เปลี่ยนสี', 'corrosion', 'rust', 'aging', 'degrad'):
        return ('Reliability', 'เสื่อมสภาพ', 'อัตโนมัติ')
    if has('รอจัดซื้อ', 'ไม่มีอะไหล่', 'อะไหล่', 'spare', 'procure', 'waiting part'):
        return ('Maintainability', 'รออะไหล่/จัดซื้อ', 'อัตโนมัติ')
    if has('trip', 'ขัดข้อง', 'ไม่ทำงาน', 'fault', 'fail', 'malfunction'):
        return ('Reliability', 'ขัดข้อง/ทำงานผิดปกติ', 'อัตโนมัติ')
    if has('ชำรุด', 'broken', 'damage', 'defect'):
        return ('Reliability', 'ชำรุด', 'อัตโนมัติ')
    return ('Maintainability', 'ไม่เข้าเงื่อนไขใด (ค่าเริ่มต้น)', 'อัตโนมัติ')

def availability_pct(r):                       # เหมือน computeAvailabilityPct
    if math.isfinite(r['mtbf']) and math.isfinite(r['mttr']) and (r['mtbf'] + r['mttr']) > 0:
        return r['mtbf'] / (r['mtbf'] + r['mttr']) * 100
    if math.isfinite(r['availability_file']):
        v = r['availability_file']
        return v * 100 if v <= 1 else v
    return float('nan')

def read_records(path):
    wb = openpyxl.load_workbook(path, data_only=True)
    recs = []
    for ws in wb.worksheets:
        rows = list(ws.iter_rows(values_only=True))
        if len(rows) < 2: continue
        head = [h if h is not None else '' for h in rows[0]]
        for row in rows[1:]:
            raw = {str(h): v for h, v in zip(head, row) if str(h) != ''}
            r = map_row(raw, ws.title)
            if not r['description'] and not r['component'] and not r['code']:
                continue
            r['equipCategory'] = equip_category(r)
            ram = parse_ram(r['ram_raw']) or auto_ram(r)
            r['ramPrimary'], r['ramSub'], r['ramSource'] = ram
            r['availabilityPct'] = availability_pct(r)
            r['raw'] = raw
            r['excluded_reason'] = ''
            r['downtimeHrs'] = float('nan')
            recs.append(r)
    return recs

# ---- 5) Data Quality Gate + MTBF/MTTR (เหมือน applyTimestampPipeline) ----
def quality_gate(recs):
    with_ts = [r for r in recs if r['failure_start'] or r['repair_complete']]
    if not with_ts:
        return {'hasTimestampData': False, 'total': 0, 'valid': 0, 'excluded': []}
    excluded, valid = [], []
    for r in with_ts:
        if not r['failure_start'] or not r['repair_complete']:
            excluded.append((r['code'], 'ข้อมูลเวลาขาดหาย (Missing time value)'))
            r['excluded_reason'] = 'ข้อมูลเวลาขาดหาย'; continue
        if r['repair_complete'] <= r['failure_start']:
            excluded.append((r['code'], 'เวลาซ่อมเสร็จเกิดก่อน/พร้อมเวลาขัดข้อง (Logical time error)'))
            r['excluded_reason'] = 'เวลาไม่สมเหตุสมผล (ซ่อมเสร็จก่อนเริ่มเสีย)'; continue
        hrs = (r['repair_complete'] - r['failure_start']).total_seconds() / 3600
        if hrs > MAX_DOWNTIME_HRS:
            excluded.append((r['code'], f'Downtime ผิดปกติ ({hrs:.0f} ชม. > 1 ปี)'))
            r['excluded_reason'] = 'Downtime ผิดปกติ (Outlier)'; continue
        r['downtimeHrs'] = hrs
        valid.append(r)
    groups = {}
    for r in valid:
        groups.setdefault((r['substation'], r['equipCategory']), []).append(r)   # แยกกลุ่มตามสถานี (ไม่มีคอลัมน์สถานี = เหมือนเดิม)
    for g in groups.values():
        g.sort(key=lambda r: r['failure_start'])
        ups = [(g[i]['failure_start'] - g[i - 1]['repair_complete']).total_seconds() / 3600 for i in range(1, len(g))]
        ups = [u for u in ups if u > 0]
        gm = sum(ups) / len(ups) if ups else float('nan')
        for r in g:
            r['mtbf'] = gm
            r['mttr'] = r['downtimeHrs']
            r['availabilityPct'] = availability_pct(r)
    return {'hasTimestampData': True, 'total': len(with_ts), 'valid': len(valid), 'excluded': excluded}

# ---- 6) สูตร R / M / A (เหมือน calcDimensions) ----
def calc_dimensions(mtbf, mttr, profile='monthly'):
    t_mission, t_repair = PROFILES[profile]
    R = math.exp(-t_mission / mtbf) * 100
    M = (1 - math.exp(-t_repair / mttr)) * 100
    A = mtbf / (mtbf + mttr) * 100
    return {'R': R, 'M': M, 'A': A, 'annualDowntime': 8760 * (1 - A / 100)}

def stats_from_occurrences(occ):               # เหมือน computeStatsFromOccurrences
    n = len(occ)
    mttr = sum(r['downtimeHrs'] for r in occ) / n if n else float('nan')
    mtbf = float('nan')
    if n >= 2:
        ups = [(occ[i]['failure_start'] - occ[i - 1]['repair_complete']).total_seconds() / 3600 for i in range(1, n)]
        ups = [u for u in ups if u > 0]
        if ups: mtbf = sum(ups) / len(ups)
    return {'mtbf': mtbf, 'mttr': mttr, 'count': n}

def summarize(recs, profile='monthly', sub=None):
    """สรุปรายประเภทอุปกรณ์ (เหมือนค่า baseline ที่หน้า What-If ใช้) — sub = เลือกเฉพาะสถานี (เหมือนปุ่มสถานีในเว็บ)"""
    cats = {}
    for r in recs:
        if sub and r['substation'] != sub: continue
        if math.isfinite(r['mtbf']) and math.isfinite(r['mttr']):
            cats.setdefault(r['equipCategory'], []).append(r)
    rows = []
    for c, rs in sorted(cats.items()):
        mtbf = sum(r['mtbf'] for r in rs) / len(rs)
        mttr = sum(r['mttr'] for r in rs) / len(rs)
        d = calc_dimensions(mtbf, mttr, profile)
        rows.append({'equipCategory': c, 'incidents': len(rs), 'MTBF_hrs': mtbf, 'MTTR_hrs': mttr,
                     'R_pct': d['R'], 'M_pct': d['M'], 'A_pct': d['A'], 'annual_downtime_hrs': d['annualDowntime'],
                     'R_pass': d['R'] >= R_STD, 'M_pass': d['M'] >= M_STD, 'A_pass': d['A'] >= A_STD})
    return rows

def code_level(recs, sub=None):
    """MTBF/MTTR รายรหัส (เหมือน computeCodeLevelStats)"""
    if sub: recs = [r for r in recs if r['substation'] == sub]
    keys = sorted({(r['equipCategory'], r['code']) for r in recs if math.isfinite(r['downtimeHrs']) and r['failure_start']})
    out = []
    for cat, code in keys:
        occ = sorted([r for r in recs if r['equipCategory'] == cat and r['code'] == code
                      and math.isfinite(r['downtimeHrs']) and r['failure_start']], key=lambda r: r['failure_start'])
        out.append({'equipCategory': cat, 'code': code, **stats_from_occurrences(occ)})
    return out


In [ ]:
# ขั้นที่ 2 — เลือกไฟล์ แล้วให้ระบบตรวจ + คำนวณ
import pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

PROFILE = 'monthly'   # 'monthly' = ภารกิจ 720 ชม. / ซ่อม 12 ชม.   |   'yearly' = 8,760 ชม. / 24 ชม.  (เหมือนปุ่มในหน้า What-If)

try:
    from google.colab import files
    up = files.upload()
    PATH = list(up.keys())[0]
except ImportError:                      # รันบนเครื่องตัวเอง
    PATH = input('พิมพ์ชื่อไฟล์ .xlsx : ').strip()

records = read_records(PATH)
gate = quality_gate(records)

print(f'อ่านได้ {len(records)} รายการ จากไฟล์ {PATH}')
if not gate['hasTimestampData']:
    print('ไม่พบคอลัมน์เวลา (เวลาเริ่มขัดข้อง / เวลาซ่อมเสร็จ) — คำนวณ MTBF/MTTR จากเวลาไม่ได้')
else:
    print(f"มีข้อมูลเวลา {gate['total']} รายการ → ใช้คำนวณได้จริง {gate['valid']} รายการ")
    if gate['excluded']:
        print(f"ตัดออก {len(gate['excluded'])} รายการ:")
        from collections import Counter
        for reason, n in Counter(e[1] for e in gate['excluded']).items():
            print(f'  - {reason}: {n} รายการ')
    else:
        print('ไม่มีรายการที่ถูกตัดออก (ข้อมูลผ่านเกณฑ์ทั้งหมด)')

summary = pd.DataFrame(summarize(records, PROFILE))
print('\n== สรุปรายประเภทอุปกรณ์ (ตัวเลขชุดนี้ต้องตรงกับหน้า What-If เมื่อเลือกประเภทเดียวกัน) ==')
display(summary.round(2)) if 'display' in globals() else print(summary.round(2))

# ถ้าไฟล์มีคอลัมน์ "สถานี" (ลาดกระบัง / ฉลองกรุง) จะสรุปแยกสถานีให้ด้วย — ตรงกับปุ่มเลือกสถานีในเว็บ
by_sub = pd.DataFrame()
subs = sorted({r['substation'] for r in records if r['substation']})
if subs:
    parts = []
    for sb in subs:
        d = pd.DataFrame(summarize(records, PROFILE, sub=sb)); d.insert(0, 'สถานี', sb); parts.append(d)
    by_sub = pd.concat(parts, ignore_index=True)
    print('\n== สรุปแยกสถานี (ตรงกับปุ่ม ลาดกระบัง / ฉลองกรุง ในเว็บ) ==')
    display(by_sub.round(2)) if 'display' in globals() else print(by_sub.round(2))

codes = pd.DataFrame(code_level(records))
print('\n== สรุปรายรหัสปัญหา (MTBF จะมีค่าเมื่อรหัสนั้นเกิดซ้ำตั้งแต่ 2 ครั้ง) ==')
display(codes.round(2)) if 'display' in globals() else print(codes.round(2))


In [ ]:
# ขั้นที่ 3 — ส่งออกไฟล์
OUT = 'RAM_Cleaned_Failure_Log.xlsx'

def fmt(d):
    return d.strftime('%d/%m/%Y %H:%M') if d else ''

ok_rows, bad_rows = [], []
for r in records:
    row = dict(r['raw'])
    if r['excluded_reason']:
        row['เหตุผลที่ตัดออก'] = r['excluded_reason']
        bad_rows.append(row)
    else:
        ok_rows.append(row)

with pd.ExcelWriter(OUT, engine='openpyxl') as xw:
    pd.DataFrame(ok_rows).to_excel(xw, sheet_name='ข้อมูลสะอาด', index=False)          # อัปโหลดเข้าเว็บได้
    pd.DataFrame(bad_rows).to_excel(xw, sheet_name='แถวที่ถูกตัดออก', index=False)
    summary.to_excel(xw, sheet_name='สรุปรายประเภท', index=False)
    codes.to_excel(xw, sheet_name='สรุปรายรหัส', index=False)
    if len(by_sub): by_sub.to_excel(xw, sheet_name='สรุปแยกสถานี', index=False)

print(f'สร้างไฟล์ {OUT} แล้ว  (สะอาด {len(ok_rows)} แถว / ตัดออก {len(bad_rows)} แถว)')
print('ชีต "ข้อมูลสะอาด" คือไฟล์ที่นำไปอัปโหลดในเว็บ RAM Insight ได้เลย')
try:
    from google.colab import files
    files.download(OUT)
except ImportError:
    pass
